GOLD

Revisando estructura de silver

In [0]:
# ============================================
# CONFIGURACIÓN GOLD
# ============================================

from pyspark.sql import functions as F

tabla_silver = "workspace.default.silver_nyctaxi"

df_silver = spark.table(tabla_silver)

print(f"Registros Silver: {df_silver.count()}")

df_silver.printSchema()

Creando indicadores diarios

In [0]:
# ============================================
# GOLD - INDICADORES DIARIOS
# ============================================

df_gold_diario = (
    df_silver
    .withColumn(
        "fecha",
        F.to_date("tpep_pickup_datetime")
    )
    .groupBy("fecha")
    .agg(
        F.count("*").alias("cantidad_viajes"),
        F.round(
            F.sum("trip_distance"), 2
        ).alias("distancia_total"),
        F.round(
            F.avg("trip_distance"), 2
        ).alias("distancia_promedio"),
        F.round(
            F.sum("fare_amount"), 2
        ).alias("ingresos_totales"),
        F.round(
            F.avg("fare_amount"), 2
        ).alias("tarifa_promedio")
    )
    .orderBy("fecha")
)

df_gold_diario.show(20, truncate=False)workspace.default.quarantine_nyctaxi

In [0]:
# ============================================
# GUARDAR GOLD - INDICADORES DIARIOS
# ============================================

(
    df_gold_diario
    .write
    .mode("overwrite")
    .saveAsTable(
        "workspace.default.gold_nyctaxi_diario"
    )
)

print("Tabla Gold diaria creada correctamente.")

Verificando guardado de gold diario

In [0]:
%sql

SELECT
    COUNT(*) AS dias,
    MIN(fecha) AS fecha_inicial,
    MAX(fecha) AS fecha_final
FROM workspace.default.gold_nyctaxi_diario;

GOLD MENSUAL

In [0]:
# ============================================
# GOLD - INDICADORES MENSUALES
# ============================================

df_gold_mensual = (
    df_silver
    .withColumn(
        "anio",
        F.year("tpep_pickup_datetime")
    )
    .withColumn(
        "mes",
        F.month("tpep_pickup_datetime")
    )
    .groupBy("anio", "mes")
    .agg(
        F.count("*").alias("cantidad_viajes"),
        F.round(
            F.sum("trip_distance"), 2
        ).alias("distancia_total"),
        F.round(
            F.avg("trip_distance"), 2
        ).alias("distancia_promedio"),
        F.round(
            F.sum("fare_amount"), 2
        ).alias("ingresos_totales"),
        F.round(
            F.avg("fare_amount"), 2
        ).alias("tarifa_promedio")
    )
    .orderBy("anio", "mes")
)

df_gold_mensual.show(truncate=False)

In [0]:
# ============================================
# GUARDAR GOLD - INDICADORES MENSUALES
# ============================================

(
    df_gold_mensual
    .write
    .mode("overwrite")
    .saveAsTable(
        "workspace.default.gold_nyctaxi_mensual"
    )
)

print("Tabla Gold mensual creada correctamente.")

In [0]:
%sql

SELECT *
FROM workspace.default.gold_nyctaxi_mensual
ORDER BY anio, mes;